# SegPlus Final End-to-End Pipeline

This notebook runs the full architecture:
- Data input and validation
- Feature engineering
- KMeans + DBSCAN + GMM modeling loop
- Cluster evaluation + stability
- Explainability (feature drivers, PCA loadings, profiles)
- Persona generation and business grounding (Ollama with fallback)
- Visualization and export


In [1]:
# Optional dependency installer/check
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    "numpy", "pandas", "scikit-learn", "matplotlib", "seaborn",
    "pyyaml", "openpyxl", "requests", "scipy"
]
OPTIONAL_PACKAGES = ["shap"]
AUTO_INSTALL_MISSING = False  # set True if you want notebook to install missing packages

missing_required = []
missing_optional = []

for pkg in REQUIRED_PACKAGES:
    mod = pkg.replace("-", "_")
    try:
        importlib.import_module(mod)
    except Exception:
        missing_required.append(pkg)

for pkg in OPTIONAL_PACKAGES:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing_optional.append(pkg)

print("Missing required:", missing_required)
print("Missing optional:", missing_optional)

if missing_required and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", *missing_required]
    print("Installing:", " ".join(missing_required))
    subprocess.check_call(cmd)


Missing required: ['scikit-learn', 'pyyaml']
Missing optional: []


In [2]:
import logging
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Ensure project root is importable when notebook is inside segplus/
cwd = Path.cwd()
project_root = cwd.parent if cwd.name.lower() == "segplus" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from segplus.config import PipelineConfig, build_domain_config_from_dataframe
from segplus.data_input import load_data, infer_schema, validate_schema
from segplus.feature_engineering import FeatureEngineer
from segplus.modeling_loop import modeling_loop
from segplus.evaluation import run_stability_test
from segplus.experiment_log import ExperimentLog
from segplus.explainability import build_explainability_report
from segplus.ollama_client import OllamaClient
from segplus.persona_generation import PersonaGenerator
from segplus.visualization import (
    plot_cluster_scatter_2d,
    plot_shap_importance,
    plot_shap_summary,
    plot_pca_loadings_heatmap,
    plot_cluster_profiles_heatmap,
    plot_cluster_sizes,
    plot_radar_charts,
    plot_experiment_history,
    plot_elbow_curve,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)-30s | %(message)s",
)

print("Project root:", project_root)


Project root: c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus


In [3]:
# Configuration (Excel direct-ingest, no domain YAML)
default_data_path = project_root / "final_enterprise_clustering_dataset_single_sheet.xlsx"
default_output_dir = project_root / "segplus_output"

config = PipelineConfig(
    data_path=str(default_data_path),
    domain_key="direct_ingest",
    sheet_name=None,
    k_range=(2, 8),
    max_iterations=5,
    pca_variance_threshold=0.85,
    output_dir=str(default_output_dir),
    ollama_host="http://localhost:11434",
    ollama_model="qwen2.5:7b",
    ollama_timeout=240,
    business_objective=(
        "Identify distinct customer segments to personalise marketing campaigns, "
        "improve retention for high-value customers, and convert mid-tier customers "
        "to premium products."
    ),
)

# Preferred model order for local Ollama auto-selection
PREFERRED_OLLAMA_MODELS = [
    "qwen2.5",
    "qwen2",
    "llama3.2",
    "llama3.1",
    "llama3",
    "mistral",
]

output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

print("Data:", config.data_path)
print("Output:", output_dir)
print("Ollama host:", config.ollama_host)
print("Primary LLM:", config.ollama_model)


Data: c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\final_enterprise_clustering_dataset_single_sheet.xlsx
Output: c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output
Ollama host: http://localhost:11434
Primary LLM: qwen2.5:7b


In [4]:
# 1) Data Input + Validation (no YAML)
df_raw = load_data(config.data_path, sheet_name=config.sheet_name)

# Build domain config directly from dataframe schema
_domain_key = config.domain_key if config.domain_key else "direct_ingest"
domain_config = build_domain_config_from_dataframe(
    df_raw,
    domain_key=_domain_key,
    exclude_cols=config.exclude_cols,
)

schema = infer_schema(df_raw)
quality_report = validate_schema(df_raw, domain_config, schema)

print(quality_report.summary())
if not quality_report.passed:
    raise ValueError("Data quality checks failed. Fix input data and rerun.")

print("Raw shape:", df_raw.shape)
print("Inferred categorical columns:", domain_config.categorical_columns[:15])


2026-03-09 12:43:26,707 | INFO     | segplus.data_input             | Loaded final_enterprise_clustering_dataset_single_sheet.xlsx: 120000 rows x 27 cols


Data Quality Report
  Rows: 120,000  |  Columns: 27
  Duplicates: 0
  Status: PASSED
Raw shape: (120000, 27)
Inferred categorical columns: ['delinquency_flag_x', 'CreditCard', 'Investment', 'Loan']


In [5]:
# 2) Feature Engineering
from sklearn.preprocessing import StandardScaler

feature_engineer = FeatureEngineer(domain_config, config)
fe_result = feature_engineer.run(df_raw)

# Original-variable matrix for SHAP (never PCA names)
original_feature_names_for_shap = fe_result.df_engineered.columns.tolist()
X_original_for_shap = StandardScaler().fit_transform(fe_result.df_engineered.values)

# Optional latent representation via local autoencoder (if tensorflow is installed)
USE_AUTOENCODER_REPRESENTATION = False
AE_LATENT_DIM = 8
AE_EPOCHS = 40
AE_BATCH_SIZE = 256

X = fe_result.X_scaled
feature_names = fe_result.feature_names
pca_for_explainability = fe_result.pca

if USE_AUTOENCODER_REPRESENTATION:
    try:
        import tensorflow as tf
        from tensorflow.keras import Model
        from tensorflow.keras.layers import Dense, Input
        from tensorflow.keras.callbacks import EarlyStopping

        tf.random.set_seed(config.random_state)
        input_dim = X.shape[1]
        latent_dim = max(2, min(AE_LATENT_DIM, input_dim - 1))

        inp = Input(shape=(input_dim,))
        x = Dense(max(16, input_dim // 2), activation="relu")(inp)
        latent = Dense(latent_dim, activation="linear", name="latent")(x)
        x = Dense(max(16, input_dim // 2), activation="relu")(latent)
        out = Dense(input_dim, activation="linear")(x)

        autoencoder = Model(inp, out)
        encoder = Model(inp, latent)
        autoencoder.compile(optimizer="adam", loss="mse")
        autoencoder.fit(
            X,
            X,
            epochs=AE_EPOCHS,
            batch_size=min(AE_BATCH_SIZE, len(X)),
            verbose=0,
            callbacks=[EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)],
        )

        X = encoder.predict(X, verbose=0)
        feature_names = [f"AE{i+1}" for i in range(X.shape[1])]
        pca_for_explainability = None  # PCA loadings no longer match AE latent space
        print(f"Autoencoder latent representation enabled: {X.shape}")
    except Exception as e:
        print(f"Autoencoder path unavailable ({e}); continuing with FE output.")

print("Engineered dataframe shape:", fe_result.df_engineered.shape)
print("Model matrix shape used for clustering:", X.shape)
print("Feature count used for clustering:", len(feature_names))
print("SHAP source feature count (original vars):", len(original_feature_names_for_shap))
print("First SHAP source vars:", original_feature_names_for_shap[:10])


2026-03-09 12:43:27,584 | INFO     | segplus.feature_engineering    | PCA: 7 components explain 87.2% variance
2026-03-09 12:43:27,585 | INFO     | segplus.feature_engineering    | Feature engineering complete: 26 features -> 7-dim output


Engineered dataframe shape: (120000, 26)
Model matrix shape used for clustering: (120000, 7)
Feature count used for clustering: 7
SHAP source feature count (original vars): 26
First SHAP source vars: ['month', 'monthly_spend_x', 'utilization_ratio_x', 'delinquency_flag_x', 'age', 'annual_income', 'risk_score', 'credit_score', 'digital_affinity', 'tenure_months']


In [6]:
# 3) Modeling Loop (KMeans + DBSCAN + GMM with reconfiguration)
experiment_log = ExperimentLog()

best_eval, best_cfg = modeling_loop(
    X=X,
    feature_names=feature_names,
    config=config,
    experiment_log=experiment_log,
)

print("Best algorithm:", best_eval.algorithm)
print("Clusters:", best_eval.n_clusters)
print("Silhouette:", best_eval.silhouette)
print("Davies-Bouldin:", best_eval.davies_bouldin)
print("Passes gate:", best_eval.passes)

exp_df = experiment_log.to_dataframe()
exp_df


c:\Users\UmairAhmed\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\UmairAhmed\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\UmairAhmed\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\UmairAhmed\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~

Best algorithm: kmeans
Clusters: 3
Silhouette: 0.2117
Davies-Bouldin: 1.5251
Passes gate: True


,iteration,timestamp,k,eps,gmm_cov,best_algorithm,n_clusters,silhouette,davies_bouldin,calinski_harabasz,passed,reconfig_strategy
0,1,2026-03-09T12:44:04.305923,3,1.661,full,kmeans,3,0.2117,1.5251,31125.2,True,None


In [7]:
# 4) Stability + Explainability
stability = run_stability_test(
    X=X,
    labels=best_eval.labels,
    n_clusters=best_eval.n_clusters,
    config=config,
    algorithm=best_eval.algorithm,
    model=best_eval.model,
)

explainability = build_explainability_report(
    X=X,
    labels=best_eval.labels,
    feature_names=feature_names,
    df_raw=fe_result.df_engineered,
    pca=pca_for_explainability,
    k_range=config.k_range,
    random_state=config.random_state,
    n_top=config.n_top_features,
    X_original=X_original_for_shap,
    original_feature_names=original_feature_names_for_shap,
)

print("Stability ARI mean:", stability.ari_mean)
print("Stability algorithm used:", best_eval.algorithm)
print("Ordered top drivers:", explainability.top_features[:10])
print("Top SHAP % (original variables):")
for k, v in list(explainability.feature_importance_pct.items())[:10]:
    print(f"  {k}: {v:.2f}%")

print("PC to original feature map (top 5):")
for pc, cols in list(explainability.pc_feature_map.items())[:5]:
    print(f"  {pc}: {cols}")

explainability.ordered_feature_drivers.head(15)

2026-03-09 12:44:11,558 | INFO     | segplus.evaluation             | Stability (kmeans): ARI=0.988 +/- 0.006 (threshold=0.70, stable=True)
2026-03-09 12:44:11,561 | INFO     | segplus.explainability         | Computing feature importances...
2026-03-09 12:44:18,121 | WARNING  | segplus.explainability         | TreeSHAP failed (only length-1 arrays can be converted to Python scalars), falling back to RF feature_importances_
2026-03-09 12:44:18,142 | INFO     | segplus.explainability         | Feature importance computed via RF feature_importances_
2026-03-09 12:44:18,146 | INFO     | segplus.explainability         | Computing PCA loadings...
2026-03-09 12:44:18,146 | INFO     | segplus.explainability         | Computing inertia curve...
2026-03-09 12:44:20,124 | INFO     | segplus.explainability         | Computing cluster profiles...
2026-03-09 12:44:20,189 | INFO     | segplus.explainability         | Building ordered convergence drivers...


Stability ARI mean: 0.988
Stability algorithm used: kmeans
Ordered top drivers: ['risk_score', 'risk_score_cluster', 'risk_behavior_score', 'delinquency_flag_y', 'credit_score', 'credit_score_cluster', 'monthly_spend_y', 'value_score', 'annual_income', 'log_income']
Top SHAP % (original variables):
  credit_score: 19.58%
  credit_score_cluster: 19.05%
  risk_score: 9.32%
  risk_score_cluster: 7.35%
  risk_behavior_score: 6.92%
  monthly_spend_y: 5.59%
  delinquency_flag_y: 5.43%
  value_score: 5.40%
  monthly_spend_x: 3.53%
  annual_income: 3.00%
PC to original feature map (top 5):
  PC1: ['credit_score', 'credit_score_cluster', 'risk_score', 'risk_score_cluster', 'log_income']
  PC2: ['value_score', 'monthly_spend_y', 'monthly_spend_x', 'lifetime_value_proxy', 'utilization_ratio_x']
  PC3: ['age', 'age_cluster', 'digital_affinity_cluster', 'digital_affinity', 'delinquency_flag_y']
  PC4: ['tenure_months', 'tenure_months_cluster', 'lifetime_value_proxy', 'utilization_ratio_x', 'utiliza

,feature,shap_importance,pca_weighted_loading,shap_rank,pca_rank,shap_rank_norm,pca_rank_norm,inertia_elbow_strength,convergence_score
0,risk_score,0.09321,0.169624,3.0,3.5,0.92,0.90,158504.140842,0.912
1,risk_score_cluster,0.07345,0.169624,4.0,3.5,0.88,0.90,158504.140842,0.888
2,risk_behavior_score,0.06918,0.170117,5.0,2.0,0.84,0.96,158504.140842,0.888
3,delinquency_flag_y,0.05429,0.170284,7.0,1.0,0.76,1.00,158504.140842,0.856
4,credit_score,0.19577,0.150041,1.0,12.5,1.00,0.54,158504.140842,0.816
5,credit_score_cluster,0.19051,0.150041,2.0,12.5,0.96,0.54,158504.140842,0.792
6,monthly_spend_y,0.05591,0.150412,6.0,10.5,0.80,0.62,158504.140842,0.728
7,value_score,0.05397,0.150412,8.0,10.5,0.72,0.62,158504.140842,0.680
8,annual_income,0.03004,0.157001,10.0,8.0,0.64,0.72,158504.140842,0.672
9,log_income,0.02988,0.159238,11.0,7.0,0.60,0.76,158504.140842,0.664


In [8]:
# 5) Persona Generation + Business Grounding (Ollama local, with fallback)
import requests


def _list_local_ollama_models(host: str) -> list[str]:
    try:
        r = requests.get(f"{host.rstrip('/')}/api/tags", timeout=8)
        r.raise_for_status()
        return [m.get("name", "") for m in r.json().get("models", []) if m.get("name")]
    except Exception as e:
        print(f"Ollama model discovery failed: {e}")
        return []


def _choose_ollama_model(installed: list[str], preferred: list[str], configured: str) -> str:
    if configured and configured in installed:
        return configured
    if configured:
        matches = [m for m in installed if configured in m]
        if matches:
            return matches[0]
    for pref in preferred:
        matches = [m for m in installed if m.startswith(pref) or pref in m]
        if matches:
            return matches[0]
    return installed[0] if installed else (configured or "qwen2.5:7b")


installed_models = _list_local_ollama_models(config.ollama_host)
selected_model = _choose_ollama_model(installed_models, PREFERRED_OLLAMA_MODELS, config.ollama_model)
config.ollama_model = selected_model

print("Installed Ollama models:", installed_models if installed_models else "None found / Ollama offline")
print("Selected model tag:", config.ollama_model)

ollama_client = OllamaClient(
    host=config.ollama_host,
    model=config.ollama_model,
    timeout=config.ollama_timeout,
)
persona_generator = PersonaGenerator(ollama_client, config)

# Step A: Cluster personas
personas = persona_generator.generate_personas(
    evaluation=best_eval,
    explainability=explainability,
    raw_df=fe_result.df_original,
    schema=schema,
)

# Step B: Business grounding + GenAI per-cluster naming/actions
grounding = persona_generator.ground_in_business_objective(personas)
personas = persona_generator.apply_grounding_to_personas(personas, grounding)

persona_df = pd.DataFrame([
    {
        "cluster": p.persona_name,
        "genai_persona_name": p.archetype,
        "profile_descriptor": getattr(p, "profile_descriptor", ""),
        "description": getattr(p, "description", ""),
        "cluster_size": p.cluster_size,
        "cluster_pct": round(p.cluster_pct, 4),
        "ordered_top_features": ", ".join([f"{k}:{v}" for k, v in p.top_features.items()]),
        "genai_recommendations": " | ".join(p.business_recommendations),
        "categorization_basis": ", ".join(getattr(p, "categorization_basis", [])),
        "naming_rationale": getattr(p, "naming_rationale", ""),
    }
    for p in personas
])

# Exact output schema check
required_persona_cols = [
    "cluster", "genai_persona_name", "profile_descriptor", "description",
    "cluster_size", "cluster_pct", "ordered_top_features",
    "genai_recommendations", "categorization_basis", "naming_rationale",
]
missing_cols = [c for c in required_persona_cols if c not in persona_df.columns]
if missing_cols:
    raise ValueError(f"persona_df missing required columns: {missing_cols}")

print("Generated personas:", len(personas))
print("Executive summary:", grounding.executive_summary)
print("persona_df columns:", list(persona_df.columns))
persona_df


Installed Ollama models: ['qwen2.5:7b']
Selected model tag: qwen2.5:7b


2026-03-09 12:44:24,482 | INFO     | segplus.ollama_client          | Ollama reachable. Model 'qwen2.5:7b' available.
2026-03-09 12:47:02,703 | INFO     | segplus.ollama_client          | Ollama response received (695 chars)
2026-03-09 12:47:26,233 | INFO     | segplus.ollama_client          | Ollama response received (12 chars)
2026-03-09 12:47:26,236 | INFO     | segplus.persona_generation     | Focused naming for Cluster A: 'High Rollers'
2026-03-09 12:47:26,247 | INFO     | segplus.persona_generation     | Persona generated: Cluster A -> High Rollers
2026-03-09 12:49:15,034 | INFO     | segplus.ollama_client          | Ollama response received (625 chars)
2026-03-09 12:49:39,299 | INFO     | segplus.ollama_client          | Ollama response received (18 chars)
2026-03-09 12:49:39,307 | INFO     | segplus.persona_generation     | Persona generated: Cluster B -> Credit Risk Strategists
2026-03-09 12:51:31,781 | INFO     | segplus.ollama_client          | Ollama response received (659 

Generated personas: 3
Executive summary: Our analysis reveals three distinct customer segments: High Rollers, Credit Risk Strategists with Higher Credit Scores, and Credit Risk Strategists with Lower Credit Scores. High Rollers represent the highest value and should be prioritized for premium product offerings. Credit Risk Strategists with Higher Credit Scores are loyal and should be retained through personalized loyalty programs. Lower Credit Score Strategists present a risk but have potential for growth through targeted credit enhancement programs.
persona_df columns: ['cluster', 'genai_persona_name', 'profile_descriptor', 'description', 'cluster_size', 'cluster_pct', 'ordered_top_features', 'genai_recommendations', 'categorization_basis', 'naming_rationale']


,cluster,genai_persona_name,profile_descriptor,description,cluster_size,cluster_pct,ordered_top_features,genai_recommendations,categorization_basis,naming_rationale
0,Cluster A,High Rollers,High Value Score & High Monthly Spend Y,High-Value Premium Spenders are among the most...,8774,0.0731,"value_score:0.1, monthly_spend_y:25414.84, mon...",Tailor premium offers and services to enhance ...,"value_score, monthly_spend_y, monthly_spend_x,...",Second-pass focused LLM naming from profile: H...
1,Cluster B,Credit Risk Strategists – Higher Credit Score ...,High Credit Score & Low Risk Score,"Creditworthy Loyalists are high-income, low-ri...",56782,0.4732,"credit_score:827.07, credit_score_cluster:827....",Target with premium product promotions | Offer...,"credit_score, credit_score_cluster, risk_score...",LLM returned generic name; using business-styl...
2,Cluster C,Credit Risk Strategists – Lower Credit Score C...,Low Credit Score & High Risk Score,These customers have slightly lower credit sco...,54444,0.4537,"credit_score:692.54, credit_score_cluster:692....",Offer personalized financial education and bud...,"credit_score, credit_score_cluster, risk_score...",LLM returned generic name; using business-styl...


In [9]:
# 6) Visualizations
plot_cluster_scatter_2d(X, best_eval.labels, output_dir, best_eval)
plot_shap_importance(
    explainability.feature_importances, output_dir, top_n=15,
    importance_pct=explainability.feature_importance_pct,
)
plot_shap_summary(
    X_original_for_shap,
    best_eval.labels,
    original_feature_names_for_shap,
    output_dir,
    random_state=config.random_state,
    max_display=15,
)
plot_pca_loadings_heatmap(explainability.pca_loadings, output_dir)
plot_cluster_profiles_heatmap(explainability.cluster_profiles, explainability.top_features, output_dir)
plot_cluster_sizes(best_eval.labels, personas, output_dir)
plot_radar_charts(fe_result.df_original, best_eval.labels, personas, explainability.top_features, output_dir)
plot_experiment_history(experiment_log, output_dir)
plot_elbow_curve(explainability.inertia_curve, output_dir)

print("Saved charts to:", output_dir)

2026-03-09 12:55:27,692 | INFO     | segplus.visualization          | Saved: cluster_scatter.png
2026-03-09 12:55:32,434 | INFO     | segplus.visualization          | Saved: feature_importance.png
2026-03-09 12:56:01,854 | INFO     | segplus.visualization          | Saved: shap_summary.png, shap_summary_bar.png, shap_interpretation.csv
2026-03-09 12:56:03,099 | INFO     | segplus.visualization          | Saved: pca_loadings.png
2026-03-09 12:56:03,653 | INFO     | segplus.visualization          | Saved: cluster_profiles.png
2026-03-09 12:56:03,848 | INFO     | segplus.visualization          | Saved: cluster_sizes.png
2026-03-09 12:56:04,758 | INFO     | segplus.visualization          | Saved: persona_radar.png
2026-03-09 12:56:05,102 | INFO     | segplus.visualization          | Saved: experiment_history.png
2026-03-09 12:56:05,498 | INFO     | segplus.visualization          | Saved: elbow_curve.png


Saved charts to: c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output


In [10]:
# 7) Export Outputs
clustered_df = fe_result.df_original.copy()
clustered_df["cluster"] = best_eval.labels

clustered_path = output_dir / "clustered_customers.csv"
profiles_path = output_dir / "cluster_profiles.csv"
ordered_drivers_path = output_dir / "ordered_feature_drivers.csv"
shap_pct_path = output_dir / "shap_feature_importance_pct.csv"
pc_feature_map_path = output_dir / "pc_feature_map.json"
personas_path = output_dir / "personas.csv"
personas_json_path = output_dir / "personas.json"
grounding_path = output_dir / "business_grounding.json"
exp_log_path = output_dir / "experiment_log.json"

clustered_df.to_csv(clustered_path, index=False)
explainability.cluster_profiles.to_csv(profiles_path)
if explainability.ordered_feature_drivers is not None:
    explainability.ordered_feature_drivers.to_csv(ordered_drivers_path, index=False)

shap_pct_df = pd.DataFrame([
    {"feature": k, "importance_pct": v}
    for k, v in explainability.feature_importance_pct.items()
]).sort_values("importance_pct", ascending=False)
shap_pct_df.to_csv(shap_pct_path, index=False)

with open(pc_feature_map_path, "w", encoding="utf-8") as f:
    json.dump(explainability.pc_feature_map, f, indent=2)

# Export exact persona schema
persona_df.to_csv(personas_path, index=False)

personas_payload = [
    {
        "cluster": p.persona_name,
        "genai_persona_name": p.archetype,
        "profile_descriptor": getattr(p, "profile_descriptor", ""),
        "description": getattr(p, "description", ""),
        "key_traits": p.key_traits,
        "genai_recommendations": p.business_recommendations,
        "cluster_size": p.cluster_size,
        "cluster_pct": p.cluster_pct,
        "top_features": p.top_features,
        "categorization_basis": getattr(p, "categorization_basis", []),
        "naming_rationale": getattr(p, "naming_rationale", ""),
    }
    for p in personas
]
with open(personas_json_path, "w", encoding="utf-8") as f:
    json.dump(personas_payload, f, indent=2)

experiment_log.to_json(exp_log_path)

with open(grounding_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "executive_summary": grounding.executive_summary,
            "cluster_priorities": grounding.cluster_priorities,
            "cluster_actions": grounding.cluster_actions,
            "quick_wins": grounding.quick_wins,
        },
        f,
        indent=2,
    )

print("Exported:")
print(" -", clustered_path)
print(" -", profiles_path)
print(" -", ordered_drivers_path)
print(" -", shap_pct_path)
print(" -", pc_feature_map_path)
print(" -", personas_path)
print(" -", personas_json_path)
print(" -", grounding_path)
print(" -", exp_log_path)


2026-03-09 12:56:08,644 | INFO     | segplus.experiment_log         | Experiment log saved: c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\experiment_log.json


Exported:
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\clustered_customers.csv
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\cluster_profiles.csv
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\ordered_feature_drivers.csv
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\shap_feature_importance_pct.csv
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\pc_feature_map.json
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\personas.csv
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\personas.json
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\business_grounding.json
 - c:\Users\UmairAhmed\OneDrive - Blend 360\Documents\Segmentation_Plus\segplus_output\experiment_log.json


## Notes

- Architecture implemented: data input -> FE -> KMeans/DBSCAN/GMM -> eval gate loop -> explainability -> ordered drivers -> persona generation -> business grounding -> per-cluster rationale outputs.
- Local Ollama is first-class. `qwen2.5` is primary model and auto-selected when installed.
- Ordered feature driver list combines SHAP importance + weighted PCA loadings, with inertia elbow context.
- `pc_feature_map.json` provides exact mapping from each PC to top original variables.
- SHAP summary plot is saved as `shap_summary.png` when `shap` is installed.
- Optional autoencoder latent representation is available via `USE_AUTOENCODER_REPRESENTATION=True`.
